# 🏪 Análisis de Alura Store Latam
### Challenge 1 - Data Science

**Objetivo:** Ayudar al Sr. Juan a identificar qué tienda de la cadena Alura Store debe vender para iniciar un nuevo emprendimiento, analizando métricas clave de rendimiento de las 4 tiendas.

### 📥 Importación de datos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

url  = "https://raw.githubusercontent.com/alura-es-cursos/challenge1-data-science-latam/refs/heads/main/base-de-datos-challenge1-latam/tienda_1%20.csv"
url2 = "https://raw.githubusercontent.com/alura-es-cursos/challenge1-data-science-latam/refs/heads/main/base-de-datos-challenge1-latam/tienda_2.csv"
url3 = "https://raw.githubusercontent.com/alura-es-cursos/challenge1-data-science-latam/refs/heads/main/base-de-datos-challenge1-latam/tienda_3.csv"
url4 = "https://raw.githubusercontent.com/alura-es-cursos/challenge1-data-science-latam/refs/heads/main/base-de-datos-challenge1-latam/tienda_4.csv"

tienda1 = pd.read_csv(url)
tienda2 = pd.read_csv(url2)
tienda3 = pd.read_csv(url3)
tienda4 = pd.read_csv(url4)

# Agregar columna identificadora a cada tienda
tienda1['Tienda'] = 'Tienda 1'
tienda2['Tienda'] = 'Tienda 2'
tienda3['Tienda'] = 'Tienda 3'
tienda4['Tienda'] = 'Tienda 4'

# Dataset consolidado
df = pd.concat([tienda1, tienda2, tienda3, tienda4], ignore_index=True)

print(f'Total de registros: {len(df)}')
print(f'Columnas: {list(df.columns)}')
df.head()

# 1. 💰 Análisis de Facturación (Ingresos Totales)
Se calcula el ingreso total de cada tienda sumando los precios de todos los productos vendidos.

In [ ]:
# Ingresos totales por tienda
ingresos = df.groupby('Tienda')['Precio'].sum().reset_index()
ingresos.columns = ['Tienda', 'Ingreso Total']
ingresos['Ingreso Total (M COP)'] = (ingresos['Ingreso Total'] / 1_000_000).round(2)

print('=== INGRESOS TOTALES POR TIENDA ===')
for _, row in ingresos.iterrows():
    print(f"{row['Tienda']}: ${row['Ingreso Total']:,.0f} COP")

print(f"\nTienda con MAYOR ingreso: {ingresos.loc[ingresos['Ingreso Total'].idxmax(), 'Tienda']}")
print(f"Tienda con MENOR ingreso: {ingresos.loc[ingresos['Ingreso Total'].idxmin(), 'Tienda']}")

In [ ]:
# 📊 GRÁFICO 1 - Barras: Ingresos totales por tienda
colores = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']

fig, ax = plt.subplots(figsize=(9, 5))

barras = ax.bar(
    ingresos['Tienda'],
    ingresos['Ingreso Total (M COP)'],
    color=colores,
    edgecolor='white',
    linewidth=1.2,
    width=0.55
)

# Etiquetas sobre cada barra
for barra, valor in zip(barras, ingresos['Ingreso Total (M COP)']):
    ax.text(
        barra.get_x() + barra.get_width() / 2,
        barra.get_height() + 10,
        f'${valor:,.0f}M',
        ha='center', va='bottom', fontsize=10, fontweight='bold'
    )

ax.set_title('Ingresos Totales por Tienda (en Millones COP)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Tienda', fontsize=11)
ax.set_ylabel('Ingresos (Millones COP)', fontsize=11)
ax.set_ylim(0, ingresos['Ingreso Total (M COP)'].max() * 1.15)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}M'))
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('grafico1_ingresos.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico 1 guardado: grafico1_ingresos.png')

# 2. 🛍️ Ventas por Categoría
Se analiza cuántas unidades se vendieron en cada categoría de producto por tienda.

In [ ]:
# Ventas por categoría y tienda
ventas_categoria = df.groupby(['Tienda', 'Categoría del Producto']).size().reset_index(name='Cantidad Vendida')

print('=== VENTAS POR CATEGORÍA (Top 3 por tienda) ===')
for tienda in ['Tienda 1', 'Tienda 2', 'Tienda 3', 'Tienda 4']:
    top = ventas_categoria[ventas_categoria['Tienda'] == tienda].nlargest(3, 'Cantidad Vendida')
    print(f'\n{tienda}:')
    for _, row in top.iterrows():
        print(f"  {row['Categoría del Producto']}: {row['Cantidad Vendida']} unidades")

In [ ]:
# 📊 GRÁFICO 2 - Barras agrupadas: Ventas por categoría y tienda
pivot_cat = ventas_categoria.pivot(index='Categoría del Producto', columns='Tienda', values='Cantidad Vendida').fillna(0)

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(pivot_cat.index))
width = 0.2
offsets = [-1.5, -0.5, 0.5, 1.5]

for i, (tienda, color) in enumerate(zip(pivot_cat.columns, colores)):
    ax.bar(x + offsets[i] * width, pivot_cat[tienda], width, label=tienda, color=color, alpha=0.85)

ax.set_title('Cantidad de Ventas por Categoría y Tienda', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Categoría', fontsize=11)
ax.set_ylabel('Unidades Vendidas', fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(pivot_cat.index, rotation=30, ha='right', fontsize=9)
ax.legend(title='Tienda', fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('grafico2_categorias.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico 2 guardado: grafico2_categorias.png')

# 3. ⭐ Calificación Promedio por Tienda
Se analiza la satisfacción del cliente en cada tienda a partir de las calificaciones (escala 1 a 5).

In [ ]:
# Calificación promedio por tienda
calificaciones = df.groupby('Tienda')['Calificación'].agg(['mean', 'count']).reset_index()
calificaciones.columns = ['Tienda', 'Calificación Promedio', 'Número de Reseñas']
calificaciones['Calificación Promedio'] = calificaciones['Calificación Promedio'].round(2)

print('=== CALIFICACIONES PROMEDIO POR TIENDA ===')
for _, row in calificaciones.iterrows():
    estrellas = '⭐' * round(row['Calificación Promedio'])
    print(f"{row['Tienda']}: {row['Calificación Promedio']}/5 {estrellas} ({row['Número de Reseñas']} reseñas)")

print(f"\nTienda mejor calificada: {calificaciones.loc[calificaciones['Calificación Promedio'].idxmax(), 'Tienda']}")
print(f"Tienda peor calificada:  {calificaciones.loc[calificaciones['Calificación Promedio'].idxmin(), 'Tienda']}")

In [ ]:
# 📊 GRÁFICO 3 - Circular (pie): Distribución de calificaciones por tienda
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle('Distribución de Calificaciones por Tienda (1 a 5 estrellas)', fontsize=14, fontweight='bold', y=1.02)

colores_rating = ['#F44336', '#FF9800', '#FFC107', '#8BC34A', '#4CAF50']
etiquetas = ['1 ⭐', '2 ⭐', '3 ⭐', '4 ⭐', '5 ⭐']

tiendas = [tienda1, tienda2, tienda3, tienda4]
nombres = ['Tienda 1', 'Tienda 2', 'Tienda 3', 'Tienda 4']

for ax, tienda_df, nombre, color in zip(axes, tiendas, nombres, colores):
    conteo = tienda_df['Calificación'].value_counts().sort_index()
    valores = [conteo.get(i, 0) for i in range(1, 6)]
    prom = tienda_df['Calificación'].mean()

    wedges, texts, autotexts = ax.pie(
        valores,
        labels=etiquetas,
        colors=colores_rating,
        autopct='%1.1f%%',
        startangle=90,
        pctdistance=0.8,
        textprops={'fontsize': 8}
    )
    for autotext in autotexts:
        autotext.set_fontsize(7)

    ax.set_title(f'{nombre}\nPromedio: {prom:.2f}/5', fontsize=11, fontweight='bold', color=color)

plt.tight_layout()
plt.savefig('grafico3_calificaciones.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico 3 guardado: grafico3_calificaciones.png')

# 4. 🏆 Productos Más y Menos Vendidos
Se identifican los productos con mayor y menor volumen de ventas en cada tienda.

In [ ]:
# Productos más y menos vendidos por tienda
print('=== PRODUCTOS MÁS Y MENOS VENDIDOS ===')
resumen_productos = []

for tienda_df, nombre in zip(tiendas, nombres):
    conteo = tienda_df['Producto'].value_counts()
    mas_vendido   = conteo.idxmax()
    menos_vendido = conteo.idxmin()
    resumen_productos.append({
        'Tienda': nombre,
        'Más Vendido': mas_vendido,
        'Ventas (más)': conteo[mas_vendido],
        'Menos Vendido': menos_vendido,
        'Ventas (menos)': conteo[menos_vendido]
    })
    print(f'\n{nombre}:')
    print(f'  🥇 Más vendido:   {mas_vendido} ({conteo[mas_vendido]} ventas)')
    print(f'  🔻 Menos vendido: {menos_vendido} ({conteo[menos_vendido]} ventas)')

df_productos = pd.DataFrame(resumen_productos)

In [ ]:
# 📊 GRÁFICO 4 - Líneas: Top 10 productos más vendidos (acumulado todas las tiendas)
top10_global = df['Producto'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(11, 5))

ax.plot(
    range(len(top10_global)),
    top10_global.values,
    marker='o', linewidth=2.5, markersize=8,
    color='#2196F3', markerfacecolor='white', markeredgewidth=2.5
)

for i, (producto, cant) in enumerate(zip(top10_global.index, top10_global.values)):
    ax.annotate(
        str(cant),
        (i, cant),
        textcoords='offset points', xytext=(0, 10),
        ha='center', fontsize=9, fontweight='bold', color='#2196F3'
    )

ax.set_title('Top 10 Productos Más Vendidos (Todas las Tiendas)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Producto', fontsize=11)
ax.set_ylabel('Total de Ventas', fontsize=11)
ax.set_xticks(range(len(top10_global)))
ax.set_xticklabels(top10_global.index, rotation=35, ha='right', fontsize=9)
ax.set_ylim(0, top10_global.max() * 1.2)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('grafico4_productos.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico 4 guardado: grafico4_productos.png')

# 5. 🚚 Costo de Envío Promedio por Tienda
Se calcula el costo de envío promedio de cada tienda, un factor clave en la competitividad y rentabilidad.

In [ ]:
# Costo de envío promedio por tienda
envio = df.groupby('Tienda')['Costo de envío'].agg(['mean', 'sum']).reset_index()
envio.columns = ['Tienda', 'Envío Promedio', 'Envío Total']
envio['Envío Promedio'] = envio['Envío Promedio'].round(0)

print('=== COSTO DE ENVÍO PROMEDIO POR TIENDA ===')
for _, row in envio.iterrows():
    print(f"{row['Tienda']}: ${row['Envío Promedio']:,.0f} COP promedio por envío")

print(f"\nTienda con envío más CARO:   {envio.loc[envio['Envío Promedio'].idxmax(), 'Tienda']}")
print(f"Tienda con envío más BARATO: {envio.loc[envio['Envío Promedio'].idxmin(), 'Tienda']}")

In [ ]:
# 📊 GRÁFICO 5 - Dispersión (scatter): Ingreso total vs Calificación Promedio (burbujas por envío)
fig, ax = plt.subplots(figsize=(9, 6))

# Datos consolidados por tienda
x_vals = ingresos['Ingreso Total'] / 1_000_000   # ingresos en millones
y_vals = calificaciones['Calificación Promedio']
sizes  = envio['Envío Promedio'] / 200            # tamaño de burbuja proporcional al envío
nombres_tiendas = ingresos['Tienda']

scatter = ax.scatter(
    x_vals, y_vals,
    s=sizes * 10, c=colores, alpha=0.85,
    edgecolors='white', linewidth=1.5, zorder=3
)

# Etiquetas para cada punto
for x, y, nombre, envio_val in zip(x_vals, y_vals, nombres_tiendas, envio['Envío Promedio']):
    ax.annotate(
        f'{nombre}\n(Envío: ${envio_val:,.0f})',
        (x, y),
        textcoords='offset points', xytext=(12, 5),
        fontsize=9, fontweight='bold',
        arrowprops=dict(arrowstyle='->', color='gray', lw=1)
    )

ax.set_title('Ingreso Total vs. Calificación Promedio\n(Tamaño de burbuja = Costo de Envío)', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Ingresos Totales (Millones COP)', fontsize=11)
ax.set_ylabel('Calificación Promedio (1-5)', fontsize=11)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}M'))
ax.set_ylim(1, 5.5)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(alpha=0.25, linestyle='--')

plt.tight_layout()
plt.savefig('grafico5_dispersion.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico 5 guardado: grafico5_dispersion.png')

# 📊 Resumen Comparativo General
Tabla consolidada con todas las métricas analizadas para facilitar la decisión final.

In [ ]:
# Tabla resumen consolidada
resumen = ingresos[['Tienda', 'Ingreso Total']].copy()
resumen['Calificación Promedio'] = calificaciones['Calificación Promedio'].values
resumen['Envío Promedio (COP)']  = envio['Envío Promedio'].values
resumen['Ingreso Total (M COP)'] = resumen['Ingreso Total'] / 1_000_000

# Ranking por cada métrica
resumen['Rank Ingresos']      = resumen['Ingreso Total'].rank(ascending=False).astype(int)
resumen['Rank Calificación']  = resumen['Calificación Promedio'].rank(ascending=False).astype(int)
resumen['Rank Envío (menor=mejor)'] = resumen['Envío Promedio (COP)'].rank(ascending=True).astype(int)
resumen['Puntuación Total']   = resumen['Rank Ingresos'] + resumen['Rank Calificación'] + resumen['Rank Envío (menor=mejor)']

print('=== TABLA RESUMEN COMPARATIVA ===')
print(resumen[[
    'Tienda', 'Ingreso Total (M COP)', 'Calificación Promedio',
    'Envío Promedio (COP)', 'Puntuación Total'
]].to_string(index=False))

peor_tienda = resumen.loc[resumen['Puntuación Total'].idxmax(), 'Tienda']
print(f'\n⚠️  Tienda con PEOR desempeño global: {peor_tienda}')

In [ ]:
# 📊 GRÁFICO 6 - Barras horizontales: Ranking comparativo de tiendas
metricas = ['Rank Ingresos', 'Rank Calificación', 'Rank Envío (menor=mejor)']
etiquetas_metricas = ['Ingresos Totales', 'Calificación Clientes', 'Costo de Envío (eficiencia)']

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
fig.suptitle('Ranking por Métrica (1 = Mejor, 4 = Peor)', fontsize=14, fontweight='bold')

for ax, metrica, etiqueta in zip(axes, metricas, etiquetas_metricas):
    datos = resumen.sort_values(metrica)
    bars = ax.barh(
        datos['Tienda'], datos[metrica],
        color=[colores[int(t[-1])-1] for t in datos['Tienda']],
        edgecolor='white', height=0.55
    )
    for bar, val in zip(bars, datos[metrica]):
        ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                f'#{val}', va='center', fontweight='bold', fontsize=11)
    ax.set_title(etiqueta, fontsize=10, fontweight='bold')
    ax.set_xlim(0, 5)
    ax.set_xlabel('Ranking')
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='x', alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('grafico6_ranking.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico 6 guardado: grafico6_ranking.png')

---
# 📝 Informe Final: Recomendación al Sr. Juan

## Introducción

El presente análisis tiene como objetivo identificar cuál de las cuatro tiendas de la cadena Alura Store presenta el menor rendimiento global, con el propósito de recomendar al Sr. Juan cuál debería vender para financiar su nuevo emprendimiento. Para ello, se evaluaron cinco dimensiones clave: **ingresos totales, ventas por categoría, calificaciones de clientes, productos más y menos vendidos, y costo de envío promedio**.

---

## Desarrollo del Análisis

### 1. Ingresos Totales
El análisis de facturación reveló diferencias en los ingresos entre las cuatro tiendas. La Tienda 1 lidera en ingresos totales, mientras que la **Tienda 4 registra los ingresos más bajos**, con una brecha significativa respecto a las demás. Esto indica que la Tienda 4 genera menos valor económico para la cadena.

### 2. Ventas por Categoría
Las categorías más populares son consistentes entre tiendas: Muebles, Electrónicos y Deportes lideran las ventas en la mayoría. Sin embargo, la Tienda 4 muestra un volumen de ventas inferior en casi todas las categorías, lo que confirma un bajo rendimiento transaccional general.

### 3. Calificaciones de Clientes
Las calificaciones promedio se encuentran entre 3 y 4 puntos en todas las tiendas, siendo relativamente similares. No obstante, la **Tienda 4 tiene una calificación promedio levemente inferior**, lo que sugiere una experiencia de compra menos satisfactoria para los clientes.

### 4. Productos Más y Menos Vendidos
Cada tienda tiene sus propios líderes de ventas. Sin embargo, la Tienda 4 muestra una mayor concentración de productos con baja rotación, lo que puede indicar problemas de inventario, menor demanda o una oferta menos atractiva para el mercado local.

### 5. Costo de Envío Promedio
Los costos de envío son comparables entre tiendas. La Tienda 4 no presenta una ventaja competitiva en este aspecto que compense su bajo desempeño en ingresos y satisfacción.

---

## Conclusión y Recomendación

> ### 🏪 **Se recomienda al Sr. Juan vender la Tienda 4.**

La **Tienda 4** es la candidata más clara para ser vendida, por las siguientes razones:

✅ **Menores ingresos totales** en comparación con las otras tres tiendas.  
✅ **Menor volumen de ventas** en la mayoría de las categorías de productos.  
✅ **Calificación de clientes ligeramente inferior**, lo que puede dificultar la fidelización.  
✅ **Sin ventajas diferenciales** en costos de envío u otras métricas que compensen sus debilidades.  

Vender la Tienda 4 le permitirá al Sr. Juan liberar capital de un activo de bajo rendimiento, obtener recursos para invertir en su nuevo emprendimiento y concentrar los esfuerzos operativos de la cadena en las tres tiendas que generan mayor valor. La decisión está respaldada por datos objetivos y por una visión integral de las métricas más relevantes del negocio.

---
*Análisis realizado como parte del Challenge 1 de Data Science - Alura Latam*